# ML2025 Homework 1 - Retrieval Augmented Generation with Agents

## Environment Setup

In [ ]:

!python3 -m pip install --no-cache-dir llama-cpp-python --extra-index-url https://abetlen.github.io/llama-cpp-python/whl/cu122
!python3 -m pip install googlesearch-python bs4 charset-normalizer requests-html lxml_html_clean

from pathlib import Path
if not Path('./Opus4.7-GODs.Ghost.Codex.Distill.4B-Q8_0.gguf').exists():
    !wget https://huggingface.co/WithinUsAI/Opus4.7-GODs.Ghost.Codex-4B.GGuF/resolve/main/Opus4.7-GODs.Ghost.Codex.Distill.4B-Q8_0.gguf
if not Path('./verilog_sample.zip').exists():
    !gdown https://drive.google.com/uc?id=1SiwEDFvVXcWlyYBw09CQOu-mhUxBB8yo
    !unzip verilog_sample.zip

Looking in indexes: https://pypi.org/simple, https://abetlen.github.io/llama-cpp-python/whl/cu122
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 GB 219.2 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 14.6 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 58.9 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.9/82.9 kB 5.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 11.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 9.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 106.8/106.8 kB 6.2 MB/s eta 0:00:00
  Attempting uninstall: websockets
    Found existing installation: websockets 14.1
    Uninstalling websockets-14.1:
      Successfully uninstalled websockets-14.1
  Attempting uninstall: urllib3
    Found existing installation: urllib3 2.3.0
   

In [2]:
import torch
if not torch.cuda.is_available():
    raise Exception('You are not using the GPU runtime. Change it first or you will suffer from the super slow inference speed!')
else:
    print('You are good to go!')

You are good to go!


## Prepare the LLM and LLM utility function

By default, we will use the quantized version of LLaMA 3.1 8B. you can get full marks on this homework by using the provided LLM and LLM utility function. You can also try out different LLM models.

In the following code block, we will load the downloaded LLM model weights onto the GPU first.
Then, we implemented the generate_response() function so that you can get the generated response from the LLM model more easily.

You can ignore "llama_new_context_with_model: n_ctx_per_seq (16384) < n_ctx_train (131072) -- the full capacity of the model will not be utilized" warning.

In [ ]:
from llama_cpp import Llama

# Load the model onto GPU
llama3 = Llama(
    "./Opus4.7-GODs.Ghost.Codex.Distill.4B-Q8_0.gguf",
    verbose=False,
    n_gpu_layers=-1,
    n_ctx=20480,    # This argument is how many tokens the model can take. The longer the better, but it will consume more memory. 16384 is a proper value for a GPU with 16GB VRAM.
)

def generate_response(_model: Llama, _messages: str) -> str:
    '''
    This function will inference the model with given messages.
    '''
    _output = _model.create_chat_completion(
        _messages,
        stop=["<|eot_id|>", "<|end_of_text|>"],
        max_tokens=16384,    # This argument is how many tokens the model can generate.
        temperature=0,      # This argument is the randomness of the model. 0 means no randomness. You will get the same result with the same input every time. You can try to set it to different values.
        repeat_penalty=2.0,
    )["choices"][0]["message"]["content"]
    return _output

## Verilog Debug Agents

The TA has implemented the Agent class for you. You can use this class to create agents that can interact with the LLM model. The Agent class has the following attributes and methods:
- Attributes:
    - role_description: The role of the agent. For example, if you want this agent to be a history expert, you can set the role_description to "You are a history expert. You will only answer questions based on what really happened in the past. Do not generate any answer if you don't have reliable sources.".
    - task_description: The task of the agent. For example, if you want this agent to answer questions only in yes/no, you can set the task_description to "Please answer the following question in yes/no. Explanations are not needed."
    - llm: Just an indicator of the LLM model used by the agent.
- Method:
    - inference: This method takes a message as input and returns the generated response from the LLM model. The message will first be formatted into proper input for the LLM model. (This is where you can set some global instructions like "Please speak in a polite manner" or "Please provide a detailed explanation".) The generated response will be returned as the output.

In [ ]:
class LLMAgent():
    def __init__(self, role_description: str, task_description: str, llm:str="bartowski/Meta-Llama-3.1-8B-Instruct-GGUF"):
        self.role_description = role_description   # Role means who this agent should act like. e.g. the history expert, the manager......
        self.task_description = task_description    # Task description instructs what task should this agent solve.
        self.llm = llm  # LLM indicates which LLM backend this agent is using.
    def inference(self, message:str) -> str:
        # TODO: Design the system prompt and user prompt here.
        # Format the messsages first.
        messages = [
            {"role": "system", "content": f"{self.role_description}"},  # Hint: you may want the agents to speak Traditional Chinese only.
            {"role": "user", "content": f"{self.task_description}\n{message}"}, # Hint: you may want the agents to clearly distinguish the task descriptions and the user messages. A proper seperation text rather than a simple line break is recommended.
        ]
        return generate_response(llama3, messages)

TODO: Define the supervisor and specialized Verilog debug agents.

In [ ]:
import re
from dataclasses import dataclass
from typing import List

DEBUG_THINKING = True


def print_agent_response(agent_name: str, text: str) -> None:
    if not DEBUG_THINKING:
        return

    print(f"\n{'=' * 80}")
    print(f"[{agent_name}]")
    print(f"{'=' * 80}")

    if text is None:
        print("(none)")
    else:
        print(text)

    print()


def clip_thinking(text: str) -> str:
    if text is None:
        return ""

    # Remove possible chain-of-thought tags
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.DOTALL)
    text = re.sub(r"```thinking.*?```", "", text, flags=re.DOTALL)

    return text.strip()


def add_line_numbers(code: str) -> str:
    return "\n".join(
        f"{idx + 1}: {line}"
        for idx, line in enumerate(code.splitlines())
    )


@dataclass
class DebugAgentSpec:
    name: str
    focus: str
    checklist: str
    examples: str = ""


def format_examples(examples: str) -> str:
    if not examples:
        return ""

    return f"\nExamples:\n{examples.strip()}\n"


class DebugAgent:
    def __init__(self, spec: DebugAgentSpec):
        self.spec = spec

        examples = format_examples(spec.examples)

        self.agent = LLMAgent(
            role_description=(
                "You are a senior RTL verification and debugging engineer.\n"
                "You specialize in ONLY ONE category of Verilog/SystemVerilog bugs.\n"
                "Your job is to detect REAL RTL bugs, not style issues.\n"
                "\n"
                "You must reason about:\n"
                "- hardware intent\n"
                "- synthesizable behavior\n"
                "- simulation correctness\n"
                "- sequential/combinational semantics\n"
                "\n"
                "Do NOT do shallow pattern matching only.\n"
                "Analyze the FULL module before making conclusions.\n"
            ),

            task_description=(
                f"You are responsible ONLY for this bug category:\n"
                f"{spec.focus}\n\n"

                "Checklist:\n"
                f"{spec.checklist}\n\n"

                "Important Rules:\n"
                "- Report ONLY bugs related to your assigned category\n"
                "- Do NOT report stylistic improvements\n"
                "- Do NOT rewrite architecture unnecessarily\n"
                "- Do NOT hallucinate bugs\n"
                "- Preserve RTL intent\n"
                "- Think like a synthesis engineer\n"
                "\n"

                "The input is line-numbered Verilog RTL.\n\n"

                "Output format:\n"
                "If no issues exist:\n"
                "No issues found\n\n"

                "Otherwise:\n"
                "Issue Type: <bug category>\n\n"
                "Location:\n"
                "Line N\n\n"
                "Why It Is Wrong:\n"
                "<hardware-level explanation>\n\n"
                "Minimal Fix:\n"
                "```verilog\n"
                "<fixed code>\n"
                "```\n\n"
                "Hardware Impact:\n"
                "<actual hardware consequence>\n\n"

                "If multiple issues exist:\n"
                "- report the MOST critical issue first\n"
                "- prioritize synthesis-breaking bugs\n"
                "\n"

                "Assume the RTL will be:\n"
                "1. compiled\n"
                "2. synthesized\n"
                "3. simulated\n\n"

                f"{examples}"
            )
        )

    def check(self, code_with_lines: str) -> str:
        raw = self.agent.inference(code_with_lines)

        print_agent_response(self.spec.name, raw)

        return clip_thinking(raw)


class SupervisorAgent:
    def __init__(self):
        self.agent = LLMAgent(
            role_description=(
                "You are the lead RTL verification and synthesis engineer.\n"
                "You supervise multiple specialized RTL debug agents.\n"
                "\n"
                "Your job is to:\n"
                "- evaluate sub-agent findings\n"
                "- reject incorrect findings\n"
                "- merge compatible fixes\n"
                "- preserve hardware intent\n"
                "- produce final corrected synthesizable RTL\n"
            ),

            task_description=(
                "You will receive:\n"
                "1. original RTL code\n"
                "2. line-numbered RTL code\n"
                "3. reports from multiple specialized agents\n\n"

                "Your responsibilities:\n"
                "- Apply ONLY valid fixes\n"
                "- Ignore hallucinated findings\n"
                "- Preserve original RTL behavior\n"
                "- Maintain module interface unless necessary\n"
                "- Keep edits minimal\n"
                "- Ensure final RTL compiles and synthesizes\n\n"

                "Priority Order:\n"
                "1. Compilation correctness\n"
                "2. Synthesis legality\n"
                "3. Sequential/combinational correctness\n"
                "4. Functional intent preservation\n"
                "5. Minimal edits\n\n"

                "Before finalizing:\n"
                "- check always blocks\n"
                "- check blocking/nonblocking semantics\n"
                "- check begin/end matching\n"
                "- check FSM completeness\n"
                "- check width consistency\n"
                "- check reset behavior\n"
                "- check inferred latches\n"
                "- check synthesizability\n\n"

                "Output ONLY the final corrected complete Verilog RTL.\n"
                "\n"
                "Output format:\n"
                "<code>\n"
                "full corrected verilog\n"
                "</code>\n"
            )
        )

    def synthesize(
        self,
        original_code: str,
        code_with_lines: str,
        findings: List[str],
    ) -> str:

        packed_findings = "\n\n".join(findings)

        prompt = (
            "Original RTL:\n"
            f"{original_code}\n\n"

            "Line-numbered RTL:\n"
            f"{code_with_lines}\n\n"

            "Sub-Agent Reports:\n"
            f"{packed_findings}\n"
        )

        raw = self.agent.inference(prompt)

        print_agent_response("supervisor", raw)

        return clip_thinking(raw)


###############################################################################
# Specialized Debug Agents
###############################################################################

debug_agents = [

    ###########################################################################
    # Syntax Agent
    ###########################################################################

    DebugAgent(
        DebugAgentSpec(
            name="syntax_agent",

            focus=(
                "Verilog syntax and parser-level errors"
            ),

            checklist=(
                "- module/endmodule syntax\n"
                "- missing semicolons\n"
                "- malformed literals\n"
                "- malformed always syntax\n"
                "- malformed assign statements\n"
                "- missing commas\n"
                "- begin/end mismatch\n"
                "- illegal declarations\n"
            ),

            examples=(
                "Input:\n"
                "1: module foo(a,b)\n"
                "2: input a;\n"
                "3: endmodule\n\n"

                "Output:\n"
                "Issue Type: syntax error\n\n"
                "Location:\n"
                "Line 1\n\n"

                "Why It Is Wrong:\n"
                "Module declaration is missing terminating semicolon.\n\n"

                "Minimal Fix:\n"
                "```verilog\n"
                "module foo(a,b);\n"
                "input a;\n"
                "endmodule\n"
                "```\n\n"

                "Hardware Impact:\n"
                "RTL fails compilation.\n"
            )
        )
    ),

    ###########################################################################
    # Sequential Logic Agent
    ###########################################################################

    DebugAgent(
        DebugAgentSpec(
            name="sequential_logic_agent",

            focus=(
                "Sequential logic bugs"
            ),

            checklist=(
                "- incorrect clock edge\n"
                "- blocking assignment inside sequential logic\n"
                "- reset logic problems\n"
                "- incomplete sequential behavior\n"
                "- multiple always block drivers\n"
                "- incorrect nonblocking usage\n"
                "- inferred latch in sequential logic\n"
            ),

            examples=(
                "Input:\n"
                "10: always @(posedge clk) begin\n"
                "11:     q = d;\n"
                "12: end\n\n"

                "Output:\n"
                "Issue Type: sequential assignment bug\n\n"

                "Location:\n"
                "Line 11\n\n"

                "Why It Is Wrong:\n"
                "Blocking assignment inside clocked sequential logic may create race conditions.\n\n"

                "Minimal Fix:\n"
                "```verilog\n"
                "always @(posedge clk) begin\n"
                "    q <= d;\n"
                "end\n"
                "```\n\n"

                "Hardware Impact:\n"
                "Simulation/synthesis mismatch may occur.\n"
            )
        )
    ),

    ###########################################################################
    # Combinational Logic Agent
    ###########################################################################

    DebugAgent(
        DebugAgentSpec(
            name="combinational_logic_agent",

            focus=(
                "Combinational logic bugs"
            ),

            checklist=(
                "- incomplete assignment\n"
                "- inferred latch\n"
                "- missing default case\n"
                "- combinational loop\n"
                "- incorrect sensitivity list\n"
                "- unintended priority logic\n"
            ),

            examples=(
                "Input:\n"
                "20: always @(*) begin\n"
                "21:     if (en)\n"
                "22:         y = a;\n"
                "23: end\n\n"

                "Output:\n"
                "Issue Type: inferred latch\n\n"

                "Location:\n"
                "Line 20\n\n"

                "Why It Is Wrong:\n"
                "Signal y is not assigned in all control paths.\n\n"

                "Minimal Fix:\n"
                "```verilog\n"
                "always @(*) begin\n"
                "    y = 0;\n"
                "    if (en)\n"
                "        y = a;\n"
                "end\n"
                "```\n\n"

                "Hardware Impact:\n"
                "Synthesis infers unintended latch hardware.\n"
            )
        )
    ),

    ###########################################################################
    # Width & Type Agent
    ###########################################################################

    DebugAgent(
        DebugAgentSpec(
            name="width_type_agent",

            focus=(
                "Signal width, signedness, and part-select bugs"
            ),

            checklist=(
                "- width mismatch\n"
                "- truncation\n"
                "- sign extension issues\n"
                "- illegal part-select\n"
                "- illegal indexing\n"
                "- concatenation width mismatch\n"
            ),

            examples=(
                "Input:\n"
                "30: wire [3:0] y;\n"
                "31: assign y = 8'hFF;\n\n"

                "Output:\n"
                "Issue Type: width truncation\n\n"

                "Location:\n"
                "Line 31\n\n"

                "Why It Is Wrong:\n"
                "8-bit value assigned into 4-bit signal truncates upper bits.\n\n"

                "Minimal Fix:\n"
                "```verilog\n"
                "wire [7:0] y;\n"
                "assign y = 8'hFF;\n"
                "```\n\n"

                "Hardware Impact:\n"
                "Upper bits are lost during assignment.\n"
            )
        )
    ),

    ###########################################################################
    # FSM Agent
    ###########################################################################

    DebugAgent(
        DebugAgentSpec(
            name="fsm_agent",

            focus=(
                "FSM state transition and encoding bugs"
            ),

            checklist=(
                "- missing state transition\n"
                "- unreachable state\n"
                "- missing default state\n"
                "- stuck FSM\n"
                "- illegal state encoding\n"
                "- incomplete case statement\n"
            ),

            examples=(
                "Input:\n"
                "40: case(state)\n"
                "41: IDLE: next_state = RUN;\n"
                "42: endcase\n\n"

                "Output:\n"
                "Issue Type: incomplete FSM case statement\n\n"

                "Location:\n"
                "Line 40\n\n"

                "Why It Is Wrong:\n"
                "FSM case statement lacks default handling.\n\n"

                "Minimal Fix:\n"
                "```verilog\n"
                "case(state)\n"
                "    IDLE: next_state = RUN;\n"
                "    default: next_state = IDLE;\n"
                "endcase\n"
                "```\n\n"

                "Hardware Impact:\n"
                "FSM may enter illegal or unknown state.\n"
            )
        )
    ),

    ###########################################################################
    # Synthesis Agent
    ###########################################################################

    DebugAgent(
        DebugAgentSpec(
            name="synthesis_agent",

            focus=(
                "Non-synthesizable RTL constructs"
            ),

            checklist=(
                "- delay operator\n"
                "- unsupported initial block\n"
                "- non-synthesizable loop\n"
                "- event control misuse\n"
                "- unsupported system task\n"
            ),

            examples=(
                "Input:\n"
                "50: #10 q <= d;\n\n"

                "Output:\n"
                "Issue Type: non-synthesizable delay\n\n"

                "Location:\n"
                "Line 50\n\n"

                "Why It Is Wrong:\n"
                "Delay operator is not synthesizable.\n\n"

                "Minimal Fix:\n"
                "```verilog\n"
                "q <= d;\n"
                "```\n\n"

                "Hardware Impact:\n"
                "RTL cannot be synthesized into hardware.\n"
            )
        )
    ),
]


###############################################################################
# Supervisor
###############################################################################

supervisor_agent = SupervisorAgent()


## Verilog Debug Pipeline

This section runs the Verilog debug pipeline and writes cleaned files (for example, code1_clean.v).

In [ ]:
def extract_code_block(text: str) -> str:
    match = re.search(r"<code>\s*(.*?)\s*</code>", text, flags=re.S | re.I)
    if not match:
        return ""
    return match.group(1).strip("\n")


def choose_output_path(src: Path) -> Path:
    stem = src.stem
    match = re.match(r"(?:code|vode)(\d+)$", stem)
    if match:
        clean_name = f"code{match.group(1)}_clean.v"
    else:
        clean_name = f"{stem}_clean.v"
    return src.with_name(clean_name)


def run_debug_flow_on_text(code: str) -> [str, str]:
    code_with_lines = add_line_numbers(code)
    findings = []
    for agent in debug_agents:
        response = agent.check(code_with_lines).strip()
        if response and response != "No issues found":
            findings.append(response)
    final_with_code = supervisor_agent.synthesize(code, code_with_lines, findings)
    final_with_code = clip_thinking(final_with_code)
    return final_with_code, extract_code_block(final_with_code)


def debug_verilog_file(file_path: Path) -> [Path, str]:
    code = file_path.read_text(encoding="utf-8", errors="ignore")
    final_with_code, cleaned_code = run_debug_flow_on_text(code)
    if not cleaned_code:
        cleaned_code = code
    out_path = choose_output_path(file_path)
    out_path.write_text(cleaned_code.rstrip() + "\n", encoding="utf-8")
    return out_path, final_with_code


def debug_all_verilog_files() -> List[[Path, Path, str]]:
    cwd = Path(".")
    candidates = []
    for path in cwd.glob("*.v"):
        if path.name.endswith("_clean.v"):
            continue
        if path.stem.startswith("code") or path.stem.startswith("vode"):
            candidates.append(path)
    results = []
    for src in sorted(candidates):
        out_path, final_with_code = debug_verilog_file(src)
        results.append((src, out_path, final_with_code))
    return results


async def pipeline(question: str = "") -> str:
    results = debug_all_verilog_files()
    if not results:
        return "No Verilog files to debug."
    lines = []
    for src, out_path, final_with_code in results:
        lines.append(f"{src.name} -> {out_path.name}")
        lines.append(final_with_code)
    return "\n\n".join(lines)


## Run the Verilog Debug Pipeline

In [ ]:
# Run the debug pipeline on all code*.v/vode*.v files in the current folder.
results_text = await pipeline()
print(results_text)

1 <think> 用户问的是「虎山雄風飛揚」是哪間學校的校歌。我需要找出這句詞出現在哪些学校的歌曲中。  首先，我可以分析這個關鍵字： - 「 tiger mountain 」  可能是指台灣南投縣埔里鎮附近的貓羅溪上游地區有座小丘稱為"大虎山" 或者更有可能的是指: 1. 臺灣省立臺東高級農業職业學校 (現國⽴台 Đông 大學附屬體育實驗中等教育機構) 20世紀5、6〇年代的校歌就有提到「巍峨的大雪山」、「雄偉虎山」  或者更精確地說： - 國立嘉義高級工業職業學校   校歌曲調為《大刀進行曲》，其中一段歌词是：   作詞者林金火先生創作時參考了當地地理特徵，將「阿里 山」與 「虎山巒 峨峰險峻」  - 國立臺南女子高級中學     也有類似的表述方式  不過最常被引用的應該還是: 國⽴嘉義⾼等⼯業學校（今為 国 站 高 工）  該校歌由林金火作詞，創作於1953年間。主要傳唱版本包含： 「虎山雄風飛揚」   這首歌曲經過多次修訂，在不同歷史時期略有調整。  目前仍舊作為该校的代表性音乐作品流传。 </think>   「**國立嘉義高級工業職業學校（簡稱：南工）**      此校歌由林金火於1953年創作，以當地地理特徵為靈感。關鍵詞「虎山」指的是該地區附近的地形特色。  ### 校歌曲調與內容 - **曲風** ：《大刀進行 曲》的變奏版    （革命時期紅歌風格）  ---  ``` 第一段： 起來！同學們，來吧！  第二節（含關鍵詞）：  虎山雄 風飛揚， 嘉南平原是我家鄉。 技術精 神發光，  建設國家為理想。  ... ```
2 <think> 這個問題是關於2015年台灣NCC（國家通訊傳播委員會）的行政命令，涉及境外郵購自用無線設備需要繳納審查費的事宜。  根據我所知： - 這項規定是在民國4年的時候實施(即西元/公元: 民102年→ 西 元：3.5)    - 精確來說是於「中華民国一百零二年度」     (也就是公历的 年)  NCC當時確實針對境外郵購自用無線通訊產品(如藍牙耳機、鍵盤等)課徵審查費用，每案收費為：  **新台幣 1,502 元**  這筆費率在後來有經過調整： - 民國4年（西元）調增至3.8萬元    - 精確來說是「中華民 国一百零四年」(公元)     (也就是 年)  因此，問題中的時間點若是2015年初(N

In [ ]:
# List generated clean files.
from pathlib import Path

clean_files = sorted(Path(".").glob("*_clean.v"))
for path in clean_files:
    print(path.name)